# BirdCLEF 2026 Inference Notebook

Ensemble + TTA + optional calibration using `birdclef_example/predict.py`.

In [ ]:
from pathlib import Path
import sys
import subprocess
import pandas as pd

# Adjust these paths for your Kaggle notebook/input names.
COMP_INPUT = Path('/kaggle/input/birdclef-2026')
MODEL_INPUT = Path('/kaggle/input/YOUR_MODEL_DATASET')
WORK_DIR = Path('/kaggle/working')

SOUNDSCAPE_DIR = COMP_INPUT / 'test_soundscapes'
METADATA_CSV = COMP_INPUT / 'sample_submission.csv'
LABEL_MAP = MODEL_INPUT / 'label_map.json'
CALIBRATION_JSON = MODEL_INPUT / 'calibration_v2.json'  # Optional

MODEL_PATHS = [
    MODEL_INPUT / 'best_model_fold1.pt',
    MODEL_INPUT / 'best_model_fold2.pt',
    MODEL_INPUT / 'best_model_fold3.pt',
    MODEL_INPUT / 'best_model_fold4.pt',
    MODEL_INPUT / 'best_model_fold5.pt',
]

OUTPUT_CSV = WORK_DIR / 'submission.csv'
TTA_OFFSETS = '0,1.25,2.5'


In [ ]:
print('Python:', sys.executable)
print('Soundscapes dir exists:', SOUNDSCAPE_DIR.exists())
print('Metadata exists:', METADATA_CSV.exists())
print('Label map exists:', LABEL_MAP.exists())
print('Calibration exists:', CALIBRATION_JSON.exists())
for p in MODEL_PATHS:
    print(p.name, p.exists())

missing = [p for p in MODEL_PATHS if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing model checkpoints: {missing}')
if not SOUNDSCAPE_DIR.exists() or not METADATA_CSV.exists() or not LABEL_MAP.exists():
    raise FileNotFoundError('One or more required input paths are missing.')

In [ ]:
cmd = [
    sys.executable, '-m', 'birdclef_example.predict',
    '--model-paths', *[str(p) for p in MODEL_PATHS],
    '--soundscape-dir', str(SOUNDSCAPE_DIR),
    '--metadata', str(METADATA_CSV),
    '--label-map', str(LABEL_MAP),
    '--tta-offsets', TTA_OFFSETS,
    '--use-bf16',
    '--output-csv', str(OUTPUT_CSV),
]

if CALIBRATION_JSON.exists():
    cmd.extend(['--calibration-json', str(CALIBRATION_JSON)])

print('Running command:')
print(' '.join(cmd))
subprocess.run(cmd, check=True)
print('Done:', OUTPUT_CSV)

In [ ]:
sample = pd.read_csv(METADATA_CSV)
sub = pd.read_csv(OUTPUT_CSV)

print('sample shape:', sample.shape)
print('submission shape:', sub.shape)

assert 'row_id' in sample.columns and 'row_id' in sub.columns
assert sub.shape[0] == sample.shape[0], 'Row count mismatch'
assert list(sub.columns) == list(sample.columns), 'Column order mismatch'
assert not sub.isna().any().any(), 'NaNs found in submission'

probs = sub.drop(columns=['row_id'])
assert ((probs >= 0.0) & (probs <= 1.0)).all().all(), 'Probabilities out of [0,1]'

print('Submission checks passed.')
sub.head()

## Submit

If checks passed, submit `/kaggle/working/submission.csv` from the notebook output panel or competition page upload.